# Tugas Praktikum

In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
import matplotlib.image as mpimg
import cv2

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

In [2]:
# 1. Klasifikasi voice dengan kernel linier, polynomial, dan RBF
voice = pd.read_csv('voice.csv')
print('Ukuran dataset:', voice.shape)
print('Jumlah nilai kosong:', voice.isna().sum().sum())
print(voice['label'].value_counts())
voice.head()

Ukuran dataset: (3168, 21)
Jumlah nilai kosong: 0
label
male      1584
female    1584
Name: count, dtype: int64


,meanfreq,sd,median,Q25,Q75,IQR,skew,kurt,sp.ent,sfm,...,centroid,meanfun,minfun,maxfun,meandom,mindom,maxdom,dfrange,modindx,label
0,0.059781,0.064241,0.032027,0.015071,0.090193,0.075122,12.863462,274.402906,0.893369,0.491918,...,0.059781,0.084279,0.015702,0.275862,0.007812,0.007812,0.007812,0.000000,0.000000,male
1,0.066009,0.067310,0.040229,0.019414,0.092666,0.073252,22.423285,634.613855,0.892193,0.513724,...,0.066009,0.107937,0.015826,0.250000,0.009014,0.007812,0.054688,0.046875,0.052632,male
2,0.077316,0.083829,0.036718,0.008701,0.131908,0.123207,30.757155,1024.927705,0.846389,0.478905,...,0.077316,0.098706,0.015656,0.271186,0.007990,0.007812,0.015625,0.007812,0.046512,male
3,0.151228,0.072111,0.158011,0.096582,0.207955,0.111374,1.232831,4.177296,0.963322,0.727232,...,0.151228,0.088965,0.017798,0.250000,0.201497,0.007812,0.562500,0.554688,0.247119,male
4,0.135120,0.079146,0.124656,0.078720,0.206045,0.127325,1.101174,4.333713,0.971955,0.783568,...,0.135120,0.106398,0.016931,0.266667,0.712812,0.007812,5.484375,5.476562,0.208274,male


In [3]:
X_voice = voice.drop(columns='label')
y_voice = voice['label']

hasil_voice_list = []
models_voice = {}
splits_voice = {}

for rasio, test_size in [('70:30', 0.3), ('80:20', 0.2)]:
    X_train_voice, X_test_voice, y_train_voice, y_test_voice = train_test_split(
        X_voice, y_voice, test_size=test_size, random_state=42, stratify=y_voice
    )
    splits_voice[rasio] = (X_train_voice, X_test_voice, y_train_voice, y_test_voice)

    for nama_kernel, kernel in [('Linier', 'linear'), ('Polynomial', 'poly'), ('RBF', 'rbf')]:
        model_voice = Pipeline([
            ('scaler', StandardScaler()),
            ('svc', SVC(kernel=kernel, C=1.0, degree=3, gamma='scale'))
        ])
        model_voice.fit(X_train_voice, y_train_voice)
        pred_train_voice = model_voice.predict(X_train_voice)
        pred_test_voice = model_voice.predict(X_test_voice)
        models_voice[(rasio, kernel)] = model_voice

        hasil_voice_list.append({
            'Rasio train:test': rasio,
            'Kernel': nama_kernel,
            'Jumlah train': len(X_train_voice),
            'Jumlah test': len(X_test_voice),
            'Akurasi train': accuracy_score(y_train_voice, pred_train_voice),
            'Akurasi test': accuracy_score(y_test_voice, pred_test_voice)
        })

hasil_voice = pd.DataFrame(hasil_voice_list)
hasil_voice

,Rasio train:test,Kernel,Jumlah train,Jumlah test,Akurasi train,Akurasi test
0,70:30,Linier,2217,951,0.972034,0.978970
1,70:30,Polynomial,2217,951,0.966622,0.960042
2,70:30,RBF,2217,951,0.983762,0.983176
3,80:20,Linier,2534,634,0.976717,0.974763
4,80:20,Polynomial,2534,634,0.968035,0.955836
5,80:20,RBF,2534,634,0.983820,0.982650


In [4]:
# 2. Klasifikasi siang dan malam dengan fitur histogram
# Gabungkan seluruh citra Praktikum 5 sebelum membagi ulang data 80:20.
daftar_citra = []
for folder_split in ['training', 'test']:
    for label, nilai_label in [('day', 1), ('night', 0)]:
        folder = Path('images') / folder_split / label
        for file in sorted(folder.glob('*.jpg')):
            daftar_citra.append({
                'Path': str(file),
                'Label': nilai_label,
                'SHA256': hashlib.sha256(file.read_bytes()).hexdigest()
            })

data_citra = pd.DataFrame(daftar_citra)
assert not data_citra.empty, 'Dataset citra tidak ditemukan.'
assert data_citra.groupby('SHA256')['Label'].nunique().max() == 1
jumlah_duplikat = data_citra.duplicated('SHA256').sum()
data_citra = data_citra.drop_duplicates('SHA256').reset_index(drop=True)

print('Jumlah citra awal:', len(daftar_citra))
print('Jumlah salinan duplikat:', jumlah_duplikat)
print('Jumlah citra unik:', len(data_citra))
print(data_citra['Label'].map({1: 'day', 0: 'night'}).value_counts())

Jumlah citra awal: 400
Jumlah salinan duplikat: 1
Jumlah citra unik: 399
Label
night    200
day      199
Name: count, dtype: int64


In [5]:
def extract_histogram(image_path, bins=32):
    image = mpimg.imread(image_path)
    image = cv2.resize(image, (1100, 600))
    image_hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)

    fitur = []
    for kanal, rentang in [(0, [0, 180]), (1, [0, 256]), (2, [0, 256])]:
        hist = cv2.calcHist([image_hsv], [kanal], None, [bins], rentang).ravel()
        hist = hist / hist.sum()
        fitur.extend(hist)
    return np.asarray(fitur, dtype=np.float32)

X_hist = np.vstack([extract_histogram(file) for file in data_citra['Path']])
y_hist = data_citra['Label'].to_numpy()

print('Fitur: histogram HSV, 32 bin per kanal, normalisasi tiap kanal.')
print('Dimensi fitur:', X_hist.shape)

Fitur: histogram HSV, 32 bin per kanal, normalisasi tiap kanal.
Dimensi fitur: (399, 96)


In [6]:
indeks_train_hist, indeks_test_hist = train_test_split(
    data_citra.index.to_numpy(), test_size=0.2, random_state=42, stratify=y_hist
)
X_train_hist = X_hist[indeks_train_hist]
X_test_hist = X_hist[indeks_test_hist]
y_train_hist = y_hist[indeks_train_hist]
y_test_hist = y_hist[indeks_test_hist]

print('Rasio train:test: 80:20')
print('Jumlah train:', len(X_train_hist))
print('Jumlah test:', len(X_test_hist))

Rasio train:test: 80:20
Jumlah train: 319
Jumlah test: 80


In [7]:
model_hist_awal = Pipeline([
    ('scaler', StandardScaler()),
    ('svc', SVC(kernel='rbf'))
])
model_hist_awal.fit(X_train_hist, y_train_hist)

cv_hist = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
param_hist = {
    'svc__C': [0.1, 1, 10, 100],
    'svc__gamma': ['scale', 0.001, 0.01, 0.1, 1]
}
grid_hist = GridSearchCV(
    model_hist_awal, param_hist, scoring='accuracy', cv=cv_hist, n_jobs=1
)
grid_hist.fit(X_train_hist, y_train_hist)
model_hist_terbaik = grid_hist.best_estimator_

print('Parameter terbaik:', grid_hist.best_params_)
print('Akurasi validasi silang terbaik:', grid_hist.best_score_)

Parameter terbaik: {'svc__C': 1, 'svc__gamma': 'scale'}
Akurasi validasi silang terbaik: 0.9905257936507936


In [8]:
hasil_tuning_hist = pd.DataFrame(grid_hist.cv_results_)
hasil_tuning_hist[['param_svc__C', 'param_svc__gamma', 'mean_test_score', 'std_test_score']].sort_values(
    'mean_test_score', ascending=False
).head(10)

,param_svc__C,param_svc__gamma,mean_test_score,std_test_score
11,10.0,0.001,0.990526,0.012674
10,10.0,scale,0.990526,0.012674
7,1.0,0.01,0.990526,0.012674
5,1.0,scale,0.990526,0.012674
16,100.0,0.001,0.990526,0.012674
17,100.0,0.01,0.990526,0.012674
15,100.0,scale,0.990526,0.012674
12,10.0,0.01,0.990526,0.012674
6,1.0,0.001,0.987450,0.006276
0,0.1,scale,0.968651,0.000198


In [9]:
hasil_hist_list = []
for nama_model, model_hist in [
    ('RBF parameter bawaan', model_hist_awal),
    ('RBF hasil tuning CV', model_hist_terbaik)
]:
    pred_train_hist = model_hist.predict(X_train_hist)
    pred_test_hist = model_hist.predict(X_test_hist)
    hasil_hist_list.append({
        'Model': nama_model,
        'Rasio train:test': '80:20',
        'Jumlah fitur': X_hist.shape[1],
        'C': model_hist.named_steps['svc'].C,
        'Gamma': model_hist.named_steps['svc'].gamma,
        'Akurasi train': accuracy_score(y_train_hist, pred_train_hist),
        'Akurasi test': accuracy_score(y_test_hist, pred_test_hist)
    })

hasil_hist = pd.DataFrame(hasil_hist_list)
hasil_hist

,Model,Rasio train:test,Jumlah fitur,C,Gamma,Akurasi train,Akurasi test
0,RBF parameter bawaan,80:20,96,1.0,scale,1.0,0.9875
1,RBF hasil tuning CV,80:20,96,1.0,scale,1.0,0.9875
